In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import sys


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer


from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score

sys.path.insert(0, r'C:\Users\Admin\Predicting-Heart-Disease')

print('Libraries imported successfully.')

Libraries imported successfully.


In [3]:
import heart_disease_lib.modeling  # import the module itself
importlib.reload(heart_disease_lib.modeling)  # reload module
from heart_disease_lib.modeling import Model, ResultAnalyzer, DataManager, CrossValidator

In [4]:
df = pd.read_csv(rf'C:\Users\Admin\Predicting-Heart-Disease\data\raw\train.csv')
print(f'Train shape: {df.shape}')

Train shape: (630000, 15)


In [5]:
X = df.drop('Heart Disease', axis=1)
y = np.where(df['Heart Disease'] == 'Presence', 1, 0)

DataMan = DataManager(X,y)

# Train Test Split
X_train, X_test, y_train, y_test = DataMan.split(X, y,test_size=0.2, random_state=42)

Train/Test Split:

X_train: (504000, 14), y_train: (504000,), y_freq: 0.45
X_test: (126000, 14), y_test: (126000,), y_freq: 0.45




# Feature Engeneering 

In [ ]:
import heart_disease_lib.modeling  # import the module itself
importlib.reload(heart_disease_lib.modeling)  # reload module
from heart_disease_lib.modeling import Model, ResultAnalyzer, DataManager, CrossValidator, FeatureEngineer

In [53]:
X_train_interactions = X_train.copy()

# 1. STRONGEST SIGNALS

# Stress test ischemia cluster
X_train_interactions["ExerciseAngina_x_STdepression"] = (X_train["Exercise angina"] * X_train["ST depression"])
X_train_interactions["STdepression_x_SlopeST"] = (X_train["ST depression"] * X_train["Slope of ST"])
X_train_interactions["ExerciseAngina_x_SlopeST"] = (X_train["Exercise angina"] * X_train["Slope of ST"])

# Disease severity imaging
X_train_interactions["Vessels_x_Thallium"] = (X_train["Number of vessels fluro"] * X_train["Thallium"])

# 2. STRONG RISK CLUSTER
# Metabolic syndrome cluster
X_train_interactions["BP_x_Cholesterol"] = (X_train["BP"] * X_train["Cholesterol"])
X_train_interactions["Cholesterol_x_FBS"] = (X_train["Cholesterol"] * X_train["FBS over 120"])
X_train_interactions["BP_x_FBS"] = (X_train["BP"] * X_train["FBS over 120"])

# 3. DEMOGRAPHIC EFFECTS
# X_train_interactions["Age_x_Sex"] = (X_train["Age"] * X_train["Sex"])
# X_train_interactions["Age_x_MaxHR"] = (X_train["Age"] * X_train["Max HR"])

# # 4. MODERATE SIGNAL
# X_train_interactions["MaxHR_x_ExerciseAngina"] = (X_train["Max HR"] * X_train["Exercise angina"])
# X_train_interactions["EKG_x_STdepression"] = (X_train["EKG results"] * X_train["ST depression"])

# # 5. WEAKER / SUPPORTIVE
# X_train_interactions["Age_x_BP"] = (X_train["Age"] * X_train["BP"])
# X_train_interactions["Age_x_Cholesterol"] = (X_train["Age"] * X_train["Cholesterol"])


In [ ]:
fe = FeatureEngineer(
    ohe_columns=['EKG results', 'Thallium'],  # One-hot encode these
    target_encode_columns=['Chest pain type'],  # Target encode these
    interaction_pairs=[
        ('Exercise angina', 'ST depression'),
        ('ST depression', 'Slope of ST'),
        ('Exercise angina', 'Slope of ST'),
        ()
    ]
)
fe.fit(X_train, y_train)
X_transformed = fe.transform(X_train)

In [ ]:
cv1 = CrossValidator(model_class=mod_xgb, data_manager_class=DataMan, k_folds=5, scoring='auc')
cv_scores1 = cv1.cross_validate(X=X_train_interactions, y=y_train, model=mod_xgb, feature_engineer=None, n_folds=5)

print(f"\nCV Scores 1: {cv_scores1}")
print(f"Mean AUC 1: {np.mean(cv_scores1):.4f} (+/- {np.std(cv_scores1):.4f})")

In [34]:
# Cross-validation with FeatureEngineer
cv1 = CrossValidator(model_class=mod_xgb, data_manager_class=DataMan, k_folds=5, scoring='auc')
cv2 = CrossValidator(model_class=mod_xgb, data_manager_class=DataMan, k_folds=5, scoring='auc')
cv3 = CrossValidator(model_class=mod_xgb, data_manager_class=DataMan, k_folds=5, scoring='auc')

# Run cross-validation with feature engineering in each fold
cv_scores1 = cv1.cross_validate(X=X_train, y=y_train, model=mod_xgb, feature_engineer=None, n_folds=5)
cv_scores2 = cv2.cross_validate(X=X_train, y=y_train, model=mod_xgb, feature_engineer=FeatureEngineer(ohe_columns=cat_cols), n_folds=5)
cv_scores3 = cv3.cross_validate(X=X_train, y=y_train, model=mod_xgb, feature_engineer=FeatureEngineer(target_encode_columns=cat_cols), n_folds=5)

print(f"\nCV Scores 1: {cv_scores1}")
print(f"Mean AUC 1: {np.mean(cv_scores1):.4f} (+/- {np.std(cv_scores1):.4f})")
print(f"\nCV Scores 2: {cv_scores2}")
print(f"Mean AUC 2: {np.mean(cv_scores2):.4f} (+/- {np.std(cv_scores2):.4f})")
print(f"\nCV Scores 3: {cv_scores3}")
print(f"Mean AUC 3: {np.mean(cv_scores3):.4f} (+/- {np.std(cv_scores3):.4f})")

[CV Fold 1] Score: 0.9536
[CV Fold 2] Score: 0.9537
[CV Fold 3] Score: 0.9545
[CV Fold 4] Score: 0.9542
[CV Fold 5] Score: 0.9543
[CV Fold 1] FeatureEngineer applied. Train shape: (403200, 23)
[CV Fold 1] Score: 0.9536
[CV Fold 2] FeatureEngineer applied. Train shape: (403200, 23)
[CV Fold 2] Score: 0.9537
[CV Fold 3] FeatureEngineer applied. Train shape: (403200, 23)
[CV Fold 3] Score: 0.9545
[CV Fold 4] FeatureEngineer applied. Train shape: (403200, 23)
[CV Fold 4] Score: 0.9542
[CV Fold 5] FeatureEngineer applied. Train shape: (403200, 23)
[CV Fold 5] Score: 0.9542
[CV Fold 1] FeatureEngineer applied. Train shape: (403200, 14)
[CV Fold 1] Score: 0.9536
[CV Fold 2] FeatureEngineer applied. Train shape: (403200, 14)
[CV Fold 2] Score: 0.9537
[CV Fold 3] FeatureEngineer applied. Train shape: (403200, 14)
[CV Fold 3] Score: 0.9545
[CV Fold 4] FeatureEngineer applied. Train shape: (403200, 14)
[CV Fold 4] Score: 0.9542
[CV Fold 5] FeatureEngineer applied. Train shape: (403200, 14)
[CV Fo

# Modeling

X_train

In [8]:
# XGBOOST MODEL
mod_xgb = Model(model = XGBClassifier(
    objective='binary:logistic',   # binary classification
    n_estimators=100,              # number of trees
    learning_rate=0.1,
    max_depth=3,
    eval_metric='logloss'          # required to suppress warning
))
#mod_xgb.fit_and_predict(X_train, y_train, X_test, y_test)

In [65]:
# Takes long time to fit. need to read avout more.
# mod_svm = Model(SVC(kernel='linear', C=1.0, random_state=42))
# mod_svm.fit(X_train, y_train)


In [38]:
# XGBOOST MODEL
mod_xgb = Model(model = XGBClassifier(
    objective='binary:logistic',   # binary classification
    n_estimators=100,              # number of trees
    learning_rate=0.1,
    max_depth=3,
    eval_metric='logloss'          # required to suppress warning
))
mod_xgb.fit_and_predict(X_train, y_train, X_test, y_test)

Model fitted


0.9538494266735198

In [43]:
# MODEL TRAINED ON ALL OF THE DATA
mod_xgb_all = Model(model = XGBClassifier(
    objective='binary:logistic',   # binary classification
    n_estimators=100,              # number of trees
    learning_rate=0.1,
    max_depth=3,
    eval_metric='logloss'          # required to suppress warning
))
mod_xgb_all.fit(X,y)

Model fitted


In [26]:
mod_lr = Model(LogisticRegression())
mod_lr.fit_and_predict(X_train, y_train, X_test, y_test)

WARNING! Data was scaled before fitting!
Model fitted
WARNING! Data was scaled before fitting!


0.9502168565640559

# Submit

In [31]:
validation_df = pd.read_csv(rf'C:\Users\Admin\Predicting-Heart-Disease\data\raw\test.csv')
print(f'Validation shape: {validation_df.shape}')

Validation shape: (270000, 14)


In [32]:
# Pre Process Validation Data
X_valid = validation_df.copy() #TODO: Replace with real pre-processing
X_Validation_scaled = scaler.fit_transform(X_valid)

## 06.02.2026

In [39]:
mod_xgb.submit(X_valid,
               submission_num='002',
               model='XGB',
               Preprocessing='none',
               notes='Baseline XGBoost model')

Submission saved & logged


In [44]:
mod_xgb_all.submit(X_valid,
               submission_num='003',
               model='XGB',
               Preprocessing='none',
               notes='Baseline XGBoost model- But trained on the entire data')

Submission saved & logged
